In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

# Load the four processed datasets
chargers = pd.read_csv(
    OUTPUT_DIR / "ev_charger_15min.csv",
    parse_dates=["timestamp"]
)

battery = pd.read_csv(
    OUTPUT_DIR / "battery_15min.csv",
    parse_dates=["timestamp"]
)

solar = pd.read_csv(
    OUTPUT_DIR / "solar_15min.csv",
    parse_dates=["timestamp"]
)

site_meter = pd.read_csv(
    OUTPUT_DIR / "site_meter_15min.csv",
    parse_dates=["timestamp"]
)

# Check for duplicate timestamps before merging
datasets = {
    "Chargers": chargers,
    "Battery": battery,
    "Solar": solar,
    "Site meter": site_meter
}

for name, df in datasets.items():
    print(
        f"{name}: {len(df)} rows | "
        f"{df['timestamp'].duplicated().sum()} duplicate timestamps"
    )

# Combine all datasets using timestamp
combined = (
    chargers
    .merge(battery, on="timestamp", how="outer")
    .merge(solar, on="timestamp", how="outer")
    .merge(site_meter, on="timestamp", how="outer")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("\nCombined dataset shape:", combined.shape)

display(combined.head())

Chargers: 20223 rows | 0 duplicate timestamps
Battery: 20256 rows | 0 duplicate timestamps
Solar: 20256 rows | 0 duplicate timestamps
Site meter: 20256 rows | 0 duplicate timestamps

Combined dataset shape: (20256, 27)


,timestamp,WB1,WB2,WB3,WB4,WB5,WB6,WB7,WB8,available_chargers,...,bat_voltage_v,reading_count_x,data_available_x,data_complete_y,solar_power_kw,reading_count_y,data_available_y,site_power_kw,reading_count,data_available
0,2025-12-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.280000,180,True,True,NaN,0,False,0.01,180,True
1,2025-12-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.279167,180,True,True,NaN,0,False,0.01,180,True
2,2025-12-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.273278,180,True,True,NaN,0,False,0.01,180,True
3,2025-12-01 00:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.271222,180,True,True,NaN,0,False,0.01,180,True
4,2025-12-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.270056,180,True,True,NaN,0,False,0.01,180,True


In [2]:
print(combined.columns.tolist())

['timestamp', 'WB1', 'WB2', 'WB3', 'WB4', 'WB5', 'WB6', 'WB7', 'WB8', 'available_chargers', 'total_ev_power_kw', 'observed_ev_power_kw', 'data_complete_x', 'observed_ev_energy_kwh', 'bat_soc_pct', 'bat_temp_c', 'bat_current_a', 'bat_voltage_v', 'reading_count_x', 'data_available_x', 'data_complete_y', 'solar_power_kw', 'reading_count_y', 'data_available_y', 'site_power_kw', 'reading_count', 'data_available']


In [3]:
combined = combined.rename(columns={
    "data_complete_x": "charger_data_complete",
    "data_complete_y": "battery_data_complete",
    "reading_count_x": "battery_reading_count",
    "data_available_x": "battery_data_available",
    "reading_count_y": "solar_reading_count",
    "data_available_y": "solar_data_available",
    "reading_count": "site_meter_reading_count",
    "data_available": "site_meter_data_available"
})

print("Updated columns:")
print(combined.columns.tolist())

print("\nDataset shape:", combined.shape)

Updated columns:
['timestamp', 'WB1', 'WB2', 'WB3', 'WB4', 'WB5', 'WB6', 'WB7', 'WB8', 'available_chargers', 'total_ev_power_kw', 'observed_ev_power_kw', 'charger_data_complete', 'observed_ev_energy_kwh', 'bat_soc_pct', 'bat_temp_c', 'bat_current_a', 'bat_voltage_v', 'battery_reading_count', 'battery_data_available', 'battery_data_complete', 'solar_power_kw', 'solar_reading_count', 'solar_data_available', 'site_power_kw', 'site_meter_reading_count', 'site_meter_data_available']

Dataset shape: (20256, 27)


In [4]:
# Count missing values in each column
missing_summary = pd.DataFrame({
    "missing_count": combined.isna().sum(),
    "missing_percentage": (
        combined.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

,missing_count,missing_percentage
timestamp,0,0.00
WB1,3022,14.92
WB2,1765,8.71
WB3,1785,8.81
WB4,1741,8.59
WB5,2169,10.71
WB6,1765,8.71
WB7,1788,8.83
WB8,2484,12.26
available_chargers,33,0.16


In [5]:
print("Start:", combined["timestamp"].min())
print("End:", combined["timestamp"].max())

print("\nTime interval check:")
print(combined["timestamp"].diff().value_counts().head())

print("\nDuplicate timestamps:", combined["timestamp"].duplicated().sum())

Start: 2025-12-01 00:00:00
End: 2026-06-29 23:45:00

Time interval check:
timestamp
0 days 00:15:00    20255
Name: count, dtype: int64

Duplicate timestamps: 0


In [6]:
output_path = OUTPUT_DIR / "integrated_microgrid_15min.csv"

combined.to_csv(output_path, index=False)

print("Exported successfully!")
print("File:", output_path)
print("Rows:", len(combined))
print("Columns:", len(combined.columns))

Exported successfully!
File: c:\Users\leena\OneDrive\Desktop\ev-microgrid-twin\data\processed\integrated_microgrid_15min.csv
Rows: 20256
Columns: 27
